# **Preprocessing**

In [1]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np
from typing import List, Dict
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("merged_data.csv.gz")

In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 435 entries, TransactionID to DeviceInfo
dtypes: float64(400), int64(4), str(31)
memory usage: 1.9 GB


In [3]:
df = df.astype({col: 'float32' for col in df.select_dtypes(include='float64').columns})

Helper function - lấy từ file eda

In [4]:
def get_feature_lists(df: pd.DataFrame, target_col: str, id_col: str | None = None) -> tuple[list[str], list[str]]:
    excluded = {target_col}
    if id_col:
        excluded.add(id_col)

    numeric_cols = [col for col in df.select_dtypes(include=np.number).columns if col not in excluded]
    categorical_cols = [col for col in df.columns if col not in excluded and col not in numeric_cols]
    return numeric_cols, categorical_cols


def build_missing_signal_table(df: pd.DataFrame, target_col: str, positive_class) -> pd.DataFrame:
    rows = []
    fraud_mask = df[target_col] == positive_class
    nonfraud_mask = df[target_col] != positive_class

    for col in df.columns:
        if col == target_col:
            continue

        is_missing = df[col].isna()
        overall_missing = is_missing.mean()

        if overall_missing == 0:
            continue

        fraud_missing = is_missing[fraud_mask].mean() if fraud_mask.any() else np.nan
        nonfraud_missing = is_missing[nonfraud_mask].mean() if nonfraud_mask.any() else np.nan
        fraud_rate_if_missing = df.loc[is_missing, target_col].eq(positive_class).mean()
        fraud_rate_if_present = df.loc[~is_missing, target_col].eq(positive_class).mean()

        rows.append(
            {
                "feature": col,
                "overall_missing_pct": overall_missing,
                "fraud_missing_pct": fraud_missing,
                "nonfraud_missing_pct": nonfraud_missing,
                "missing_gap": fraud_missing - nonfraud_missing if pd.notna(fraud_missing) and pd.notna(nonfraud_missing) else np.nan,
                "abs_missing_gap": abs(fraud_missing - nonfraud_missing) if pd.notna(fraud_missing) and pd.notna(nonfraud_missing) else np.nan,
                "fraud_rate_if_missing": fraud_rate_if_missing,
                "fraud_rate_if_present": fraud_rate_if_present,
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return result.sort_values("abs_missing_gap", ascending=False).reset_index(drop=True)

## **1. Handle missing value**

- Các cột missing value ~ 99% và không có class signal, drop cột
- Các cột numerical có abs_missing_gap >= 0.02 và overall_missing_pct >= 0.05 -> tạo cột isna và fill giá trị -999 để mô hình học được đây là giá trị bất thường
- Cột categorical fill giá trị 'missing' vào NaN

In [5]:
cols_to_drop = ['id_24', 'id_25', 'id_07', 'id_08', 'id_21','id_26','id_22','id_23','id_27']

df.drop(columns=cols_to_drop, inplace=True)

In [6]:
class Numeric_Missing_Processor(BaseEstimator, TransformerMixin):
    """
    - Tạo cột _isna CHỈ cho numerical có missing gap hoặc missing rate cao
    - Impute giá trị -999 để mô hình cây hiểu được bản chất bất thường
    """
    def __init__(self, 
                 gap_threshold: float = 0.02,      # abs_missing_gap >= 2%
                 missing_threshold: float = 0.05,  # missing_pct >= 5%
                 fill_value: float = -999.0): 
        self.gap_threshold = gap_threshold
        self.missing_threshold = missing_threshold
        self.fill_value = fill_value
        self.indicator_cols: List[str] = []       # danh sách cột sẽ tạo _isna
        self.missing_signal_ = None

    def fit(self, X: pd.DataFrame, y=None):
        """Fit trên train set (có target)"""
        target_col = 'isFraud'
        positive_class = 1
        
        self.missing_signal_ = build_missing_signal_table(X, target_col, positive_class)
        
        for _, row in self.missing_signal_.iterrows():
            col = row['feature']
            if col == target_col:
                continue
            gap = row.get('abs_missing_gap', 0)
            miss_pct = row.get('overall_missing_pct', 0)
        
            if pd.api.types.is_numeric_dtype(X[col]) and \
               (gap >= self.gap_threshold or miss_pct >= self.missing_threshold):
                self.indicator_cols.append(col)
        
        print(f"[Numeric] Tạo missing indicator cho {len(self.indicator_cols)} cột")
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        cols = [col for col in self.indicator_cols if col in X.columns]
        if cols:
            isna_df = pd.DataFrame({
                f"{col}_isna": X[col].isna().astype("int8")
                for col in cols
            })
            X = pd.concat([X, isna_df], axis=1)
            X[cols] = X[cols].fillna(self.fill_value)
        return X



class Categorical_Missing_Processor(BaseEstimator, TransformerMixin):
    """
    - Giữ nguyên các category phổ biến (Common).
    - Gán NaN thực sự thành nhãn "MISSING".
    - Các giá trị còn lại (quá hiếm) gán thành "RARE"
    """
    def __init__(self, cat_cols: List[str], min_freq: float = 0.001):
        self.cat_cols = cat_cols
        self.min_freq = min_freq
        self.freq_maps_: Dict[str, List] = {}

    def fit(self, X: pd.DataFrame, y=None):
        for col in self.cat_cols:
            if col in X.columns:
                freq = X[col].value_counts(normalize=True, dropna=False)
                common = freq[freq >= self.min_freq].index.tolist()
                self.freq_maps_[col] = common
        print(f"[Categorical] Xử lý {len(self.cat_cols)} cột categorical")
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(object)
                
                is_missing = X[col].isna()
                
                common = self.freq_maps_.get(col, [])
                is_common = X[col].isin(common)
                
                X[col] = np.where(is_missing, "MISSING", 
                                 np.where(is_common, X[col], "RARE"))
                
                X[col] = X[col].astype("category")
        return X

In [7]:
numeric_cols, categorical_cols = get_feature_lists(df, target_col='isFraud', id_col='TransactionID')

# Khởi tạo processors
numeric_processor = Numeric_Missing_Processor(
    fill_value=-999.0
)

cat_processor = Categorical_Missing_Processor(
    cat_cols=categorical_cols,
    min_freq=0.0005           # < 0.1% coi là rare
)

numeric_processor.fit(df)
cat_processor.fit(df)

df_eda_clean = numeric_processor.transform(df)
df_eda_clean = cat_processor.transform(df_eda_clean)

print(f"✅ Hoàn tất! Số cột trước: {df.shape[1]} → sau: {df_eda_clean.shape[1]}")


print("\n📊 Missing indicators được tạo:")
print([col for col in df_eda_clean.columns if col.endswith('_isna')][:20])  # top 20

# Lưu processors để dùng sau (production / inference)
# import joblib
# joblib.dump(numeric_processor, "models/preprocessors/numeric_missing_processor.pkl")
# joblib.dump(cat_processor, "models/preprocessors/cat_missing_processor.pkl")

[Numeric] Tạo missing indicator cho 287 cột
[Categorical] Xử lý 29 cột categorical
✅ Hoàn tất! Số cột trước: 426 → sau: 713

📊 Missing indicators được tạo:
['id_02_isna', 'id_11_isna', 'id_01_isna', 'id_05_isna', 'id_06_isna', 'V200_isna', 'V188_isna', 'V189_isna', 'V194_isna', 'V195_isna', 'V184_isna', 'V201_isna', 'V197_isna', 'V198_isna', 'V175_isna', 'V210_isna', 'V208_isna', 'V209_isna', 'V170_isna', 'V185_isna']


In [8]:
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols: List[str]):
        self.cat_cols = cat_cols
        self.encoders_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        for col in self.cat_cols:
            if col in X.columns:
                le = LabelEncoder()
                # Ép kiểu string để tránh lỗi mix types
                le.fit(X[col].astype(str))
                self.encoders_[col] = le
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        for col, le in self.encoders_.items():
            if col in X.columns:
                # Xử lý các giá trị mới xuất hiện ở test set (nếu có) bằng cách map về 'RARE'
                X[col] = X[col].astype(str).map(lambda s: s if s in le.classes_ else 'RARE')
                X[col] = le.transform(X[col])
        return X

In [9]:
# Giả sử bạn đã có df_eda_clean từ bước trước đó
# Bước này tiếp nối cell cuối cùng trong preprocessing.ipynb của bạn

# 1. Khởi tạo encoder với danh sách cột categorical đã lấy từ get_feature_lists
label_encoder = CategoricalEncoder(cat_cols=categorical_cols)

# 2. Fit trên dữ liệu đã được xử lý Missing/Rare
# Lưu ý: Phải fit trên dữ liệu đã qua bước Categorical_Missing_Processor 
# để encoder "học" được cả nhãn 'MISSING' và 'RARE'
label_encoder.fit(df_eda_clean)

# 3. Chuyển đổi dữ liệu sang dạng số
df_encoded = label_encoder.transform(df_eda_clean)

# Kiểm tra kết quả
print(f"Hình dạng dữ liệu sau encoding: {df_encoded.shape}")
display(df_encoded[categorical_cols].head())

Hình dạng dữ liệu sau encoding: (590540, 713)


,ProductCD,card4,card6,P_emaildomain,R_emaildomain,M1,M2,M3,M4,M5,...,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,4,2,2,0,0,2,2,2,2,0,...,7,0,24,0,1,1,1,1,0,1
1,4,3,2,12,0,0,1,1,0,2,...,7,0,24,0,1,1,1,1,0,1
2,4,4,3,24,0,2,2,2,0,0,...,7,0,24,0,1,1,1,1,0,1
3,4,3,3,31,0,0,1,1,0,2,...,7,0,24,0,1,1,1,1,0,1
4,1,3,2,12,0,0,1,1,3,1,...,3,38,16,4,2,0,2,2,2,3


In [10]:
# import joblib
# joblib.dump(label_encoder, "categorical_encoder.pkl")

In [11]:
class SkewedFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, log_cols: List[str] = None, clip_percentile: float = 0.99):
        self.log_cols = log_cols if log_cols else []
        self.clip_percentile = clip_percentile
        self.clip_thresholds_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        # Xác định ngưỡng cắt (clipping) cho các cột số
        numeric_cols = X.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            # Tính toán phân vị (ví dụ 99%) để làm ngưỡng cắt
            self.clip_thresholds_[col] = X[col].quantile(self.clip_percentile)
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        # 1. Clipping có kiểm soát: Giới hạn các giá trị cực đại
        for col, threshold in self.clip_thresholds_.items():
            if col in X.columns:
                X[col] = X[col].clip(upper=threshold)
        
        # 2. Log-like transform: Dùng log1p (log(1+x)) cho nhóm cực lệch
        # giúp xử lý tốt cả những điểm "spike-at-zero"
        for col in self.log_cols:
            if col in X.columns:
                X[col] = np.log1p(X[col])
        return X

In [12]:
class CategoricalLevelManager(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols: List[str], min_freq: float = 0.001):
        self.cat_cols = cat_cols
        self.min_freq = min_freq
        self.common_labels_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        for col in self.cat_cols:
            if col in X.columns:
                # Tính tần suất của từng nhãn
                freq = X[col].value_counts(normalize=True, dropna=True)
                # Chỉ giữ lại các nhãn có tần suất > min_freq
                common = freq[freq >= self.min_freq].index.tolist()
                self.common_labels_[col] = common
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        for col in self.cat_cols:
            if col in X.columns:
                # Chuyển về kiểu chuỗi để xử lý nhất quán
                series = X[col].astype(str)
                
                # Xác định các vị trí thiếu dữ liệu (NaN ban đầu)
                is_null = X[col].isna() | (series == 'nan') | (series == 'None')
                
                # Xác định các nhãn phổ biến (đã học từ bước fit)
                common = self.common_labels_.get(col, [])
                is_common = series.isin(common)
                
                # Áp dụng logic: 
                # Nếu trống -> MISSING
                # Nếu không phổ biến -> OTHER
                # Còn lại -> Giữ nguyên
                X[col] = np.where(is_null, "MISSING",
                                 np.where(is_common, series, "OTHER"))
        return X

In [13]:
numeric_cols, categorical_cols = get_feature_lists(df, target_col='isFraud')

# Khởi tạo các bước xử lý mới
skew_transformer = SkewedFeatureTransformer(
    log_cols=['TransactionAmt', 'C1', 'C2'], # Ví dụ các cột cực lệch từ EDA
    clip_percentile=0.99
)

cat_manager = CategoricalLevelManager(
    cat_cols=categorical_cols,
    min_freq=0.0005 # Tần suất tối thiểu để không bị coi là 'OTHER'
)

# Thực thi tuần tự
df_step1 = skew_transformer.fit_transform(df)
df_final = cat_manager.fit_transform(df_step1)

df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 426 entries, TransactionID to DeviceInfo
dtypes: float32(393), float64(2), int64(2), str(29)
memory usage: 1.0 GB
